In [ ]:
# Environment check. Prints everything mounted under /kaggle/input so we can
# confirm the EEG datasets attached before the pipeline runs.

import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


In [1]:
import os

# Walk the input tree until we hit a folder that actually holds EDF
# recordings plus a summary file. That folder is the CHB-MIT root.
for root, dirs, files in os.walk('/kaggle/input'):
    edf = [f for f in files if f.endswith('.edf')]
    summary = [f for f in files if 'summary' in f.lower() or f.endswith('.txt')]
    if edf or summary:
        print(root)
        print("  EDF files:", len(edf), "| example:", edf[:2])
        print("  summary/txt:", summary[:3])
        break  # drop this break to list every patient folder

/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0
  EDF files: 0 | example: []
  summary/txt: ['SHA256SUMS.txt']


In [2]:
import os

BASE = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'

# Step 1 - raw data inventory. Patient folders are named chb01 .. chb24.
patient_dirs = sorted([d for d in os.listdir(BASE) if d.startswith('chb')])
print("Patient folders:", patient_dirs)
print("Count:", len(patient_dirs))

# Spot-check one patient so we know the folder layout
if patient_dirs:
    p0 = os.path.join(BASE, patient_dirs[0])
    files = sorted(os.listdir(p0))
    edf = [f for f in files if f.endswith('.edf')]
    summary = [f for f in files if 'summary' in f.lower()]
    print(f"\n{patient_dirs[0]}: {len(edf)} EDF files")
    print("  first EDFs:", edf[:3])
    print("  summary file:", summary)

# Grand total of recordings in the database
total_edf = sum(len([f for f in os.listdir(os.path.join(BASE, d)) if f.endswith('.edf')]) for d in patient_dirs)
print("\nTotal EDF files:", total_edf)

Patient folders: ['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb08', 'chb09', 'chb10', 'chb11', 'chb12', 'chb13', 'chb14', 'chb15', 'chb16', 'chb17', 'chb18', 'chb19', 'chb20', 'chb21', 'chb22', 'chb23', 'chb24']
Count: 24

chb01: 42 EDF files
  first EDFs: ['chb01_01.edf', 'chb01_02.edf', 'chb01_03.edf']
  summary file: ['chb01-summary.txt']

Total EDF files: 686


In [3]:
# Utility: zip the figures folder so it can be pulled off the session at once.
import shutil
shutil.make_archive('/kaggle/working/all_figures', 'zip', '/kaggle/working/figures')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/figures'

In [4]:
import os
import re

BASE = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'

def parse_summary(patient):
    """Read chbXX-summary.txt and return one record per EDF file,
    each carrying that file's seizure start/end times."""
    summary_path = os.path.join(BASE, patient, f'{patient}-summary.txt')
    with open(summary_path, 'r') as f:
        text = f.read()

    records = []
    # The summary is one block per EDF file; split on the file-name marker
    blocks = re.split(r'File Name:', text)
    for block in blocks[1:]:
        fname_match = re.match(r'\s*(\S+\.edf)', block)
        if not fname_match:
            continue
        fname = fname_match.group(1)

        # Seizure count for this file (0 for the clean ones)
        nsz_match = re.search(r'Number of Seizures in File:\s*(\d+)', block)
        n_sz = int(nsz_match.group(1)) if nsz_match else 0

        # Pull every start/end pair - one file can carry more than one seizure
        starts = [int(x) for x in re.findall(r'Seizure(?:\s+\d+)?\s+Start Time:\s*(\d+)\s*seconds', block)]
        ends   = [int(x) for x in re.findall(r'Seizure(?:\s+\d+)?\s+End Time:\s*(\d+)\s*seconds', block)]
        seizures = list(zip(starts, ends))

        records.append({
            'patient': patient,
            'file': fname,
            'n_seizures': n_sz,
            'seizures': seizures,  # list of (start_sec, end_sec) within this file
        })
    return records

# Check the parser against chb01, where the seizure-bearing files are known
recs = parse_summary('chb01')
print(f"chb01: {len(recs)} files parsed")
for r in recs:
    if r['n_seizures'] > 0:
        print(f"  {r['file']}: {r['n_seizures']} seizure(s) -> {r['seizures']}")

# chb01 is documented with 7 seizures. Anything else means the regex is off.
total_sz = sum(r['n_seizures'] for r in recs)
print(f"\nchb01 total seizures: {total_sz} (expected 7)")

chb01: 42 files parsed
  chb01_03.edf: 1 seizure(s) -> [(2996, 3036)]
  chb01_04.edf: 1 seizure(s) -> [(1467, 1494)]
  chb01_15.edf: 1 seizure(s) -> [(1732, 1772)]
  chb01_16.edf: 1 seizure(s) -> [(1015, 1066)]
  chb01_18.edf: 1 seizure(s) -> [(1720, 1810)]
  chb01_21.edf: 1 seizure(s) -> [(327, 420)]
  chb01_26.edf: 1 seizure(s) -> [(1862, 1963)]

chb01 total seizures: 7 (expected 7)


In [5]:
import json

# Step 2 - run the parser over every patient and tally seizures.
inventory = {}
for patient in sorted([d for d in os.listdir(BASE) if d.startswith('chb')]):
    try:
        recs = parse_summary(patient)
    except FileNotFoundError:
        print(f"  WARNING: no summary for {patient}, skipping")
        continue
    total_sz = sum(r['n_seizures'] for r in recs)
    sz_files = [r['file'] for r in recs if r['n_seizures'] > 0]
    inventory[patient] = {
        'total_seizures': total_sz,
        'n_seizure_files': len(sz_files),
        'seizure_files': sz_files,
        'all_records': recs,
    }

# Per-patient table, flagging who clears the three-seizure bar
print(f"{'Patient':<10}{'Seizures':<10}{'SzFiles':<10}{'Eligible(>=3)'}")
print("-" * 45)
eligible = []
for p, info in inventory.items():
    elig = info['total_seizures'] >= 3
    if elig:
        eligible.append(p)
    print(f"{p:<10}{info['total_seizures']:<10}{info['n_seizure_files']:<10}{'YES' if elig else 'no'}")

print("-" * 45)
print(f"\nEligible patients (>=3 seizures): {len(eligible)}")
print(eligible)

# Cache to disk so later stages never re-parse the summary files
os.makedirs('/kaggle/working/artifacts', exist_ok=True)
with open('/kaggle/working/artifacts/inventory.json', 'w') as f:
    # all_records is bulky and only needed in-session, so save the slim version
    slim = {p: {k: v for k, v in info.items() if k != 'all_records'} for p, info in inventory.items()}
    slim['_eligible'] = eligible
    json.dump(slim, f, indent=2)
print("\nSaved: /kaggle/working/artifacts/inventory.json")

Patient   Seizures  SzFiles   Eligible(>=3)
---------------------------------------------
chb01     7         7         YES
chb02     3         3         YES
chb03     7         7         YES
chb04     4         3         YES
chb05     5         5         YES
chb06     10        7         YES
chb07     3         3         YES
chb08     5         5         YES
chb09     4         3         YES
chb10     7         7         YES
chb11     3         3         YES
chb12     40        13        YES
chb13     12        8         YES
chb14     8         7         YES
chb15     20        14        YES
chb16     10        6         YES
chb17     3         3         YES
chb18     6         6         YES
chb19     3         3         YES
chb20     8         6         YES
chb21     4         4         YES
chb22     3         3         YES
chb23     7         3         YES
chb24     16        12        YES
---------------------------------------------

Eligible patients (>=3 seizures): 24
['chb01', 

In [6]:
import os, json, numpy as np, mne
mne.set_log_level('ERROR')  # otherwise MNE logs a line per file

BASE = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'

# ------------- Labelling parameters, fixed for the whole study -------------
SPH_MIN   = 5      # seizure prediction horizon (discarded gap before onset)
SOP_MIN   = 30     # seizure occurrence period (pre-ictal window)
INTERICTAL_GAP_H = 4   # interictal must be >=4h from any seizure on/offset
WIN_SEC   = 30     # window length, non-overlapping
FS        = 256    # CHB-MIT native sampling rate
BANDPASS  = (0.5, 50.0)
NOTCH     = 60.0   # US mains
# ---------------------------------------------------------------------------

def get_file_start_seconds(patient, records):
    """Kept as a stub on purpose.

    The summary files give wall-clock times, so stitching every recording into
    one continuous timeline means trusting those clocks across file gaps. We
    found that fragile. Instead each file is handled on its own: windows are
    labelled against the seizures inside that same file, and interictal data is
    only drawn from files that contain no seizure at all."""
    return None  # see label_file below for the per-file approach we use instead

def label_file(patient, rec, raw_dur_sec):
    """Label every 30 s window of one EDF file.

    Returns (win_start_sec, win_end_sec, label) triples, where label is one of
    'preictal', 'interictal' or 'discard', plus a flag for whether this file
    contained a seizure."""
    seizures = rec['seizures']  # [(start,end),...] in seconds within this file
    labels = []
    n_wins = int(raw_dur_sec // WIN_SEC)

    for w in range(n_wins):
        ws, we = w*WIN_SEC, (w+1)*WIN_SEC
        wc = (ws+we)/2  # window center
        lab = 'interictal_candidate'  # default; downgraded below

        if seizures:
            for (sz_s, sz_e) in seizures:
                preictal_start = sz_s - (SPH_MIN+SOP_MIN)*60
                preictal_end   = sz_s - SPH_MIN*60
                sph_start      = sz_s - SPH_MIN*60
                # the SOP window we want to forecast from
                if preictal_start <= wc < preictal_end:
                    lab = 'preictal'; break
                # the SPH gap just before onset: thrown away on purpose
                if sph_start <= wc < sz_s:
                    lab = 'discard'; break
                # seizure itself plus the post-ictal recovery buffer
                if sz_s <= wc < sz_e + INTERICTAL_GAP_H*3600:
                    lab = 'discard'; break
                # run-up to the preictal window: too close to call either way
                if preictal_start - INTERICTAL_GAP_H*3600 <= wc < preictal_start:
                    lab = 'discard'; break
        labels.append((ws, we, lab))
    return labels, (len(seizures) > 0)

# ---- Eyeball the labelling on one seizure file and one clean file ----
inv = json.load(open('/kaggle/working/artifacts/inventory.json'))
recs01 = parse_summary('chb01')

test_files = ['chb01_03.edf', 'chb01_01.edf']  # first has a seizure, second does not
for fname in test_files:
    rec = next(r for r in recs01 if r['file'] == fname)
    path = os.path.join(BASE, 'chb01', fname)
    raw = mne.io.read_raw_edf(path, preload=False, verbose='ERROR')
    dur = raw.n_times / raw.info['sfreq']
    labels, has_sz = label_file('chb01', rec, dur)
    from collections import Counter
    c = Counter(l[2] for l in labels)
    print(f"{fname}: dur={dur:.0f}s, seizures={rec['seizures']}, has_sz={has_sz}")
    print(f"  window labels: {dict(c)}")
    if 'preictal' in c:
        pre = [l for l in labels if l[2]=='preictal']
        print(f"  preictal spans: {pre[0][0]}s -> {pre[-1][1]}s (should end ~{rec['seizures'][0][0]-SPH_MIN*60}s)")
    print()

chb01_03.edf: dur=3600s, seizures=[(2996, 3036)], has_sz=True
  window labels: {'discard': 60, 'preictal': 60}
  preictal spans: 900s -> 2700s (should end ~2696s)

chb01_01.edf: dur=3600s, seizures=[], has_sz=False
  window labels: {'interictal_candidate': 120}



In [7]:
import os, json, numpy as np, mne
from scipy import signal as sps
from scipy.stats import entropy as scipy_entropy
from collections import Counter
mne.set_log_level('ERROR')

BASE = '/kaggle/input/datasets/abhishekinnvonix/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'
OUT = '/kaggle/working/artifacts'
os.makedirs(OUT, exist_ok=True)

SPH_MIN, SOP_MIN, INTERICTAL_GAP_H = 5, 30, 4
WIN_SEC, FS = 30, 256
BANDS = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,50)}
CLEAN_FILE_BUFFER = 2  # a clean file this close to a seizure file is still not trusted as interictal

def common_channels(patient, sz_files_any):
    """Channels present in every one of a patient's recordings.

    Montages drift between files, so we take the intersection and work with
    that fixed set throughout."""
    pdir = os.path.join(BASE, patient)
    edfs = sorted([f for f in os.listdir(pdir) if f.endswith('.edf')])
    ch_sets = []
    for f in edfs:
        try:
            raw = mne.io.read_raw_edf(os.path.join(pdir, f), preload=False)
            ch_sets.append(set(raw.ch_names))
        except Exception:
            continue
    common = set.intersection(*ch_sets) if ch_sets else set()
    # keep only bipolar EEG channels (contain '-'), drop dummy/status channels
    common = sorted([c for c in common if '-' in c and c.upper() not in ('.-.','-')])
    return common

def extract_features(win_data, fs=FS):
    """Flatten one (n_channels, n_samples) window into a feature vector."""
    feats = []
    for ch in win_data:
        # relative band power from a Welch periodogram
        f, pxx = sps.welch(ch, fs=fs, nperseg=min(256, len(ch)))
        total = np.trapz(pxx, f) + 1e-12
        for (lo, hi) in BANDS.values():
            idx = (f >= lo) & (f < hi)
            feats.append(np.trapz(pxx[idx], f[idx]) / total)
        # line length: total sample-to-sample movement
        feats.append(np.sum(np.abs(np.diff(ch))))
        # spectral entropy of the normalised spectrum
        p = pxx / (pxx.sum() + 1e-12)
        feats.append(scipy_entropy(p + 1e-12))
    return np.array(feats, dtype=np.float32)

def process_patient(patient):
    recs = parse_summary(patient)
    chans = common_channels(patient, None)
    if len(chans) < 16:
        return None  # too few channels, skip (report it)

    # summary order is chronological, so keep it
    file_order = [r['file'] for r in recs]
    sz_file_idx = set(i for i,r in enumerate(recs) if r['n_seizures'] > 0)
    # only clean files far enough from any seizure file qualify as interictal
    interictal_ok = set()
    for i, r in enumerate(recs):
        if r['n_seizures'] == 0:
            near = any(abs(i-j) <= CLEAN_FILE_BUFFER for j in sz_file_idx)
            if not near:
                interictal_ok.add(r['file'])

    X, y, seizure_id, seg_type = [], [], [], []
    sz_counter = 0
    for r in recs:
        fpath = os.path.join(BASE, patient, r['file'])
        try:
            raw = mne.io.read_raw_edf(fpath, preload=True)
        except Exception:
            continue
        raw.pick_channels(chans, ordered=True)
        raw.filter(0.5, 50., verbose='ERROR')
        try: raw.notch_filter(60., verbose='ERROR')
        except Exception: pass
        data = raw.get_data()
        dur = data.shape[1] / FS
        labels, has_sz = label_file(patient, r, dur)

        # tag each preictal window with its seizure, so splits can stay seizure-wise
        for (ws, we, lab) in labels:
            s0, s1 = int(ws*FS), int(we*FS)
            if s1 > data.shape[1]:
                continue
            if lab == 'preictal':
                win = data[:, s0:s1]
                X.append(extract_features(win)); y.append(1)
                # seizure id is a running counter over the patient's seizures
                seizure_id.append(f"{patient}_sz{sz_counter}")
                seg_type.append('preictal')
            elif lab == 'interictal_candidate' and r['file'] in interictal_ok:
                win = data[:, s0:s1]
                X.append(extract_features(win)); y.append(0)
                seizure_id.append(f"{patient}_inter")
                seg_type.append('interictal')
        if has_sz:
            sz_counter += r['n_seizures']

    if not X or sum(y) == 0:
        return None
    return {
        'patient': patient, 'channels': chans,
        'X': np.array(X), 'y': np.array(y),
        'seizure_id': np.array(seizure_id), 'seg_type': np.array(seg_type),
    }

# ---- Feature extraction across the whole database ----
eligible = json.load(open(f'{OUT}/inventory.json'))['_eligible']
report = {}
for p in eligible:
    print(f"Processing {p}...", flush=True)
    res = process_patient(p)
    if res is None:
        report[p] = 'SKIPPED (channels/seizures)'
        print(f"  {p}: skipped"); continue
    np.savez_compressed(f'{OUT}/{p}_features.npz',
                        X=res['X'], y=res['y'],
                        seizure_id=res['seizure_id'], seg_type=res['seg_type'])
    n_pre, n_int = int((res['y']==1).sum()), int((res['y']==0).sum())
    report[p] = {'n_channels': len(res['channels']),
                 'n_preictal': n_pre, 'n_interictal': n_int,
                 'n_unique_seizures': len(set(res['seizure_id'])) - (1 if n_int>0 else 0)}
    print(f"  {p}: {len(res['channels'])}ch, pre={n_pre}, inter={n_int}")

json.dump(report, open(f'{OUT}/feature_report.json','w'), indent=2)
print("\nDONE. Report saved to feature_report.json")
print(json.dumps(report, indent=2))

Processing chb01...


/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(pxx, f) + 1e-12
/tmp/ipykernel_58/2264031937.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  feats.append(np.trapz(pxx[idx], f[idx]) / total)
/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(pxx, f) + 1e-12
/tmp/ipykernel_58/2264031937.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  feats.append(np.trapz(pxx[idx], f[idx]) / total)
/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration 

  chb01: 23ch, pre=271, inter=2400
Processing chb02...


/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(pxx, f) + 1e-12
/tmp/ipykernel_58/2264031937.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  feats.append(np.trapz(pxx[idx], f[idx]) / total)
/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(pxx, f) + 1e-12
/tmp/ipykernel_58/2264031937.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  feats.append(np.trapz(pxx[idx], f[idx]) / total)
/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration 

  chb02: 23ch, pre=120, inter=3240
Processing chb03...


...

  chb22: 28ch, pre=152, inter=1920
Processing chb23...


/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(pxx, f) + 1e-12
/tmp/ipykernel_58/2264031937.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  feats.append(np.trapz(pxx[idx], f[idx]) / total)
/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(pxx, f) + 1e-12
/tmp/ipykernel_58/2264031937.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  feats.append(np.trapz(pxx[idx], f[idx]) / total)
/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration 

  chb23: 23ch, pre=121, inter=1065
Processing chb24...


/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(pxx, f) + 1e-12
/tmp/ipykernel_58/2264031937.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  feats.append(np.trapz(pxx[idx], f[idx]) / total)
/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  total = np.trapz(pxx, f) + 1e-12
/tmp/ipykernel_58/2264031937.py:41: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  feats.append(np.trapz(pxx[idx], f[idx]) / total)
/tmp/ipykernel_58/2264031937.py:38: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration 

  chb24: 23ch, pre=466, inter=0

DONE. Report saved to feature_report.json
{
  "chb01": {
    "n_channels": 23,
    "n_preictal": 271,
    "n_interictal": 2400,
    "n_unique_seizures": 7
  },
  "chb02": {
    "n_channels": 23,
    "n_preictal": 120,
    "n_interictal": 3240,
    "n_unique_seizures": 2
  },
  "chb03": {
    "n_channels": 23,
    "n_preictal": 243,
    "n_interictal": 3000,
    "n_unique_seizures": 7
  },
  "chb04": {
    "n_channels": 23,
    "n_preictal": 166,
    "n_interictal": 12873,
    "n_unique_seizures": 3
  },
  "chb05": {
    "n_channels": 23,
    "n_preictal": 210,
    "n_interictal": 2400,
    "n_unique_seizures": 5
  },
  "chb06": {
    "n_channels": 23,
    "n_preictal": 295,
    "n_interictal": 0,
    "n_unique_seizures": 7
  },
  "chb07": {
    "n_channels": 23,
    "n_preictal": 180,
    "n_interictal": 4159,
    "n_unique_seizures": 3
  },
  "chb08": {
    "n_channels": 23,
    "n_preictal": 299,
    "n_interictal": 600,
    "n_unique_seizures": 5
  }

In [8]:
import json, numpy as np, os
OUT = '/kaggle/working/artifacts'
report = json.load(open(f'{OUT}/feature_report.json'))

# Step 3 - final cohort. A patient is usable only if both classes survived
# the labelling rules above.
cohort = [p for p,v in report.items()
          if isinstance(v, dict) and v['n_interictal'] > 0 and v['n_preictal'] > 0]
excluded = [p for p in report if p not in cohort]

print(f"FINAL COHORT: N={len(cohort)}")
print(cohort)
print(f"\nEXCLUDED (N={len(excluded)}): {excluded}")

# How lopsided each patient's class balance ended up
print(f"\n{'Patient':<9}{'Pre':<7}{'Inter':<8}{'Ratio':<10}{'Seizures':<9}{'Ch'}")
print("-"*50)
for p in cohort:
    v = report[p]
    ratio = v['n_interictal'] / max(v['n_preictal'],1)
    print(f"{p:<9}{v['n_preictal']:<7}{v['n_interictal']:<8}1:{ratio:<8.1f}{v['n_unique_seizures']:<9}{v['n_channels']}")

json.dump({'cohort':cohort,'excluded':excluded}, open(f'{OUT}/cohort.json','w'), indent=2)
print("\nSaved cohort.json")

FINAL COHORT: N=21
['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb07', 'chb08', 'chb09', 'chb10', 'chb11', 'chb13', 'chb14', 'chb15', 'chb16', 'chb17', 'chb18', 'chb19', 'chb20', 'chb21', 'chb22', 'chb23']

EXCLUDED (N=3): ['chb06', 'chb12', 'chb24']

Patient  Pre    Inter   Ratio     Seizures Ch
--------------------------------------------------
chb01    271    2400    1:8.9     7        23
chb02    120    3240    1:27.0    2        23
chb03    243    3000    1:12.3    7        23
chb04    166    12873   1:77.5    3        23
chb05    210    2400    1:11.4    5        23
chb07    180    4159    1:23.1    3        23
chb08    299    600     1:2.0     5        23
chb09    180    4079    1:22.7    3        23
chb10    396    2160    1:5.5     7        23
chb11    98     3599    1:36.7    2        23
chb13    234    2040    1:8.7     7        21
chb14    342    720     1:2.1     7        28
chb15    351    360     1:1.0     12       30
chb16    180    840     1:4.7     5        21
chb1

In [9]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)

OUT = '/kaggle/working/artifacts'
cohort = json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d = np.load(f'{OUT}/{p}_features.npz', allow_pickle=True)
    return d['X'], d['y'], d['seizure_id'], d['seg_type']

def conformal_fnr_threshold(scores_pre_cal, alpha):
    """Split-conformal threshold aimed at holding the miss rate at or below alpha.

    scores_pre_cal holds P(preictal) for the calibration preictal windows. We
    raise an alarm when a score reaches tau, so a preictal window is missed
    whenever its score falls short. Putting tau at the alpha-quantile of the
    calibration scores (with the finite-sample correction) is what bounds that
    miss rate."""
    n = len(scores_pre_cal)
    if n == 0:
        return 0.0
    # finite-sample corrected quantile level
    q_level = np.floor(alpha * (n + 1)) / n
    q_level = min(max(q_level, 0.0), 1.0)
    tau = np.quantile(scores_pre_cal, q_level, method='lower')
    return tau

def eval_fnr_fpr(model, scaler, X_pre, X_int, tau):
    fnr = fpr = np.nan
    if len(X_pre):
        s = model.predict_proba(scaler.transform(X_pre))[:,1]
        fnr = np.mean(s < tau)         # missed preictal
    if len(X_int):
        s = model.predict_proba(scaler.transform(X_int))[:,1]
        fpr = np.mean(s >= tau)        # false alarms on interictal
    return fnr, fpr

ALPHA = 0.10  # target: miss at most 10% of preictal windows
results = {'random':{}, 'time':{}}

for p in cohort:
    X, y, sid, seg = load_patient(p)
    pre_mask, int_mask = (y==1), (y==0)
    Xp, Xi = X[pre_mask], X[int_mask]
    sid_p = sid[pre_mask]
    uniq_sz = sorted(set(sid_p.tolist()))
    if len(uniq_sz) < 2:
        continue

    for split_type in ['random','time']:
        if split_type == 'time':
            # chronological split: earliest half of the seizures calibrate, rest are test
            k = len(uniq_sz); n_cal = int(np.ceil(k/2))
            cal_sz, test_sz = set(uniq_sz[:n_cal]), set(uniq_sz[n_cal:])
        else:
            # same split size, but seizures drawn at random
            perm = list(np.random.permutation(uniq_sz))
            n_cal = int(np.ceil(len(perm)/2))
            cal_sz, test_sz = set(perm[:n_cal]), set(perm[n_cal:])

        cal_pre = Xp[np.isin(sid_p, list(cal_sz))]
        test_pre = Xp[np.isin(sid_p, list(test_sz))]
        # interictal split the same way: first/second half for time, shuffled for random
        ni = len(Xi)
        if split_type=='time':
            cut = ni//2; cal_int, test_int = Xi[:cut], Xi[cut:]
        else:
            idx = np.random.permutation(ni); cut=ni//2
            cal_int, test_int = Xi[idx[:cut]], Xi[idx[cut:]]

        if len(cal_pre)<5 or len(test_pre)<1:
            continue

        # ---- backbone classifier, trained only on the calibration seizures ----
        n_bal = min(len(cal_pre)*3, len(cal_int))
        cal_int_bal = cal_int[np.random.choice(len(cal_int), n_bal, replace=False)]
        Xtr = np.vstack([cal_pre, cal_int_bal])
        ytr = np.r_[np.ones(len(cal_pre)), np.zeros(len(cal_int_bal))]
        scaler = StandardScaler().fit(Xtr)
        model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(scaler.transform(Xtr), ytr)

        # ---- the obvious conformal threshold, taken from calibration preictal scores ----
        s_cal_pre = model.predict_proba(scaler.transform(cal_pre))[:,1]
        tau = conformal_fnr_threshold(s_cal_pre, ALPHA)

        # ---- evaluate on seizures the model has never seen ----
        fnr, fpr = eval_fnr_fpr(model, scaler, test_pre, test_int, tau)
        results[split_type][p] = {'realized_fnr':float(fnr), 'realized_fpr':float(fpr),
                                  'target_fnr':ALPHA, 'n_cal_pre':int(len(cal_pre)),
                                  'n_test_pre':int(len(test_pre))}

json.dump(results, open(f'{OUT}/stage5_naive.json','w'), indent=2)

# ---- does the guarantee survive the transfer? ----
for st in ['random','time']:
    r = results[st]
    fnrs = [v['realized_fnr'] for v in r.values() if not np.isnan(v['realized_fnr'])]
    violations = sum(f > ALPHA + 0.02 for f in fnrs)  # small tolerance
    print(f"\n[{st.upper()} split]  target FNR={ALPHA}")
    print(f"  patients: {len(fnrs)}, mean realized FNR: {np.mean(fnrs):.3f}")
    print(f"  GUARANTEE VIOLATIONS (FNR>target): {violations}/{len(fnrs)}")
    print(f"  worst-patient FNR: {np.max(fnrs):.3f}")


[RANDOM split]  target FNR=0.1
  patients: 21, mean realized FNR: 0.754
  GUARANTEE VIOLATIONS (FNR>target): 20/21
  worst-patient FNR: 1.000

[TIME split]  target FNR=0.1
  patients: 21, mean realized FNR: 0.699
  GUARANTEE VIOLATIONS (FNR>target): 21/21
  worst-patient FNR: 1.000


In [10]:
import numpy as np, json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

for p in ['chb15','chb04']:  # the most and least balanced patients
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]
    print(f"\n=== {p} ===")
    print(f"  X shape {X.shape}, preictal {len(Xp)}, interictal {len(Xi)}")
    print(f"  X any NaN? {np.isnan(X).any()}  any Inf? {np.isinf(X).any()}")
    print(f"  X range: min {X.min():.3g}, max {X.max():.3g}, mean {X.mean():.3g}")

    # quick window-level split, purely to confirm the features separate at all
    n=min(len(Xp),len(Xi))
    idxp=np.random.permutation(len(Xp))[:n]; idxi=np.random.permutation(len(Xi))[:n]
    Xall=np.vstack([Xp[idxp],Xi[idxi]]); yall=np.r_[np.ones(n),np.zeros(n)]
    perm=np.random.permutation(2*n); Xall,yall=Xall[perm],yall[perm]
    cut=int(0.6*len(Xall))
    sc=StandardScaler().fit(Xall[:cut])
    m=LogisticRegression(max_iter=1000,class_weight='balanced').fit(sc.transform(Xall[:cut]),yall[:cut])
    s=m.predict_proba(sc.transform(Xall[cut:]))[:,1]
    yt=yall[cut:]
    print(f"  train acc {m.score(sc.transform(Xall[:cut]),yall[:cut]):.3f}")
    print(f"  test preictal-score mean {s[yt==1].mean():.3f} vs interictal-score mean {s[yt==0].mean():.3f}")
    print(f"  --> if these two means are close, features aren't separating classes")


=== chb15 ===
  X shape (711, 210), preictal 351, interictal 360
  X any NaN? False  any Inf? False
  X range: min 8.57e-35, max 4.86, mean 0.536
  train acc 0.974
  test preictal-score mean 0.875 vs interictal-score mean 0.170
  --> if these two means are close, features aren't separating classes

=== chb04 ===
  X shape (13039, 161), preictal 166, interictal 12873
  X any NaN? False  any Inf? False
  X range: min 0.000264, max 3.94, mean 0.509
  train acc 1.000
  test preictal-score mean 0.915 vs interictal-score mean 0.142
  --> if these two means are close, features aren't separating classes


In [11]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

def conformal_fnr_threshold(scores_pre_cal, alpha):
    """Same threshold as above, written against the order statistics directly.

    Preictal windows score high and we alarm at tau, so a miss is a score
    below tau. Taking the floor(alpha*(n+1))-th smallest calibration score is
    the conformal way to bound that miss probability by alpha."""
    s = np.sort(np.asarray(scores_pre_cal))   # ascending
    n = len(s)
    if n == 0: return 0.0
    # largest tau that still keeps at most an alpha fraction underneath it
    k = int(np.floor(alpha * (n + 1)))         # conformal rank
    k = min(max(k, 0), n)                       # clamp
    if k == 0:
        return s[0] - 1e-9                       # threshold below all -> FNR ~0
    return s[k-1]                               # k-th smallest score

def eval_fnr_fpr(model, scaler, X_pre, X_int, tau):
    fnr=fpr=np.nan
    if len(X_pre):
        s=model.predict_proba(scaler.transform(X_pre))[:,1]; fnr=np.mean(s<tau)
    if len(X_int):
        s=model.predict_proba(scaler.transform(X_int))[:,1]; fpr=np.mean(s>=tau)
    return fnr,fpr

ALPHA=0.10
results={'random':{},'time':{}}
for p in cohort:
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<2: continue
    for st in ['random','time']:
        if st=='time':
            k=len(uniq); nc=int(np.ceil(k/2))
            cal_sz,test_sz=set(uniq[:nc]),set(uniq[nc:])
        else:
            perm=list(np.random.permutation(uniq)); nc=int(np.ceil(len(perm)/2))
            cal_sz,test_sz=set(perm[:nc]),set(perm[nc:])
        cal_pre=Xp[np.isin(sid_p,list(cal_sz))]; test_pre=Xp[np.isin(sid_p,list(test_sz))]
        ni=len(Xi)
        if st=='time': cut=ni//2; cal_int,test_int=Xi[:cut],Xi[cut:]
        else:
            idx=np.random.permutation(ni); cut=ni//2
            cal_int,test_int=Xi[idx[:cut]],Xi[idx[cut:]]
        if len(cal_pre)<5 or len(test_pre)<1: continue
        n_bal=min(len(cal_pre)*3,len(cal_int))
        cbi=cal_int[np.random.choice(len(cal_int),n_bal,replace=False)]
        Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
        sc=StandardScaler().fit(Xtr)
        m=LogisticRegression(max_iter=1000,class_weight='balanced').fit(sc.transform(Xtr),ytr)
        s_cal=m.predict_proba(sc.transform(cal_pre))[:,1]
        tau=conformal_fnr_threshold(s_cal,ALPHA)
        fnr,fpr=eval_fnr_fpr(m,sc,test_pre,test_int,tau)
        results[st][p]={'realized_fnr':float(fnr),'realized_fpr':float(fpr),
                        'target':ALPHA,'n_cal_pre':int(len(cal_pre)),'tau':float(tau)}
json.dump(results,open(f'{OUT}/stage5_naive.json','w'),indent=2)

for st in ['random','time']:
    r=results[st]; fnrs=[v['realized_fnr'] for v in r.values() if not np.isnan(v['realized_fnr'])]
    fprs=[v['realized_fpr'] for v in r.values() if not np.isnan(v['realized_fpr'])]
    viol=sum(f>ALPHA+0.02 for f in fnrs)
    print(f"\n[{st.upper()}] target FNR={ALPHA}")
    print(f"  patients {len(fnrs)}, mean FNR {np.mean(fnrs):.3f}, worst {np.max(fnrs):.3f}")
    print(f"  violations {viol}/{len(fnrs)}, mean FPR (alarm burden) {np.mean(fprs):.3f}")


[RANDOM] target FNR=0.1
  patients 21, mean FNR 0.754, worst 1.000
  violations 20/21, mean FPR (alarm burden) 0.015

[TIME] target FNR=0.1
  patients 21, mean FNR 0.699, worst 1.000
  violations 21/21, mean FPR (alarm burden) 0.173


In [14]:
import numpy as np, json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

p='chb15'
X,y,sid,seg=load_patient(p)
Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
uniq=sorted(set(sid_p.tolist()))
perm=list(np.random.permutation(uniq)); nc=int(np.ceil(len(perm)/2))
cal_sz,test_sz=set(perm[:nc]),set(perm[nc:])
cal_pre=Xp[np.isin(sid_p,list(cal_sz))]; test_pre=Xp[np.isin(sid_p,list(test_sz))]
ni=len(Xi); idx=np.random.permutation(ni); cut=ni//2
cal_int,test_int=Xi[idx[:cut]],Xi[idx[cut:]]

n_bal=min(len(cal_pre)*3,len(cal_int))
cbi=cal_int[np.random.choice(len(cal_int),n_bal,replace=False)]
Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
sc=StandardScaler().fit(Xtr)
m=LogisticRegression(max_iter=1000,class_weight='balanced').fit(sc.transform(Xtr),ytr)

s_cal=m.predict_proba(sc.transform(cal_pre))[:,1]
s_test_pre=m.predict_proba(sc.transform(test_pre))[:,1]
s_test_int=m.predict_proba(sc.transform(test_int))[:,1]

alpha=0.10
s=np.sort(s_cal); n=len(s)
k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
tau=s[k-1] if k>0 else s[0]-1e-9

print(f"cal preictal scores: n={n}")
print(f"  min {s.min():.3f}, 10th pct {np.percentile(s,10):.3f}, median {np.median(s):.3f}, max {s.max():.3f}")
print(f"conformal k={k}, tau={tau:.4f}")
print(f"test preictal scores: mean {s_test_pre.mean():.3f}, frac < tau (=FNR) {np.mean(s_test_pre<tau):.3f}")
print(f"test interict scores: mean {s_test_int.mean():.3f}, frac >= tau (=FPR) {np.mean(s_test_int>=tau):.3f}")
print(f"\nsample cal preictal scores: {np.round(s[:10],3)}")
print(f"sample test preictal scores: {np.round(s_test_pre[:10],3)}")

cal preictal scores: n=178
  min 0.444, 10th pct 0.774, median 0.978, max 1.000
conformal k=17, tau=0.7656
test preictal scores: mean 0.648, frac < tau (=FNR) 0.399
test interict scores: mean 0.130, frac >= tau (=FPR) 0.033

sample cal preictal scores: [0.444 0.461 0.56  0.633 0.644 0.665 0.68  0.686 0.693 0.695]
sample test preictal scores: [0.009 0.007 0.007 0.003 0.006 0.039 0.    0.809 0.061 0.349]


In [21]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

ALPHA=0.10

def static_conformal_tau(scores_pre, alpha):
    s=np.sort(scores_pre); n=len(s)
    if n==0: return 0.0
    k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
    return (s[0]-1e-9) if k==0 else s[k-1]

def train_backbone(cal_pre, cal_int):
    n_bal=min(len(cal_pre)*3,len(cal_int))
    cbi=cal_int[np.random.choice(len(cal_int),n_bal,replace=False)]
    Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
    sc=StandardScaler().fit(Xtr)
    m=LogisticRegression(max_iter=1000,class_weight='balanced').fit(sc.transform(Xtr),ytr)
    return m,sc

def run_patient(p, adaptive, gamma=0.05):
    """Walk a patient's held-out seizures in the order they happened.

    static  - tau is fixed once from the calibration seizures.
    adaptive - tau moves after each seizure in response to the error so far,
               the usual online-conformal update."""
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<3: return None            # need >=1 cal + >=2 test seizures for adaptation
    k=len(uniq); nc=max(1,int(np.ceil(k/2)))
    cal_sz, test_sz = uniq[:nc], uniq[nc:]  # TIME-ordered
    cal_pre=Xp[np.isin(sid_p,cal_sz)]
    if len(cal_pre)<5: return None
    ni=len(Xi); cut=ni//2; cal_int,test_int_all=Xi[:cut],Xi[cut:]
    m,sc=train_backbone(cal_pre,cal_int)

    s_cal=m.predict_proba(sc.transform(cal_pre))[:,1]
    tau=static_conformal_tau(s_cal,ALPHA)   # both start from conformal tau

    # interictal scores, split across the test seizures for false-alarm accounting
    s_int_test=m.predict_proba(sc.transform(test_int_all))[:,1]
    int_chunks=np.array_split(s_int_test, len(test_sz))

    fnr_seq, fpr_seq, tau_seq = [], [], []
    for j, szid in enumerate(test_sz):
        tp=Xp[sid_p==szid]
        s=m.predict_proba(sc.transform(tp))[:,1]
        fnr = np.mean(s<tau) if len(s) else np.nan
        fpr = np.mean(int_chunks[j]>=tau) if len(int_chunks[j]) else np.nan
        fnr_seq.append(fnr); fpr_seq.append(fpr); tau_seq.append(tau)
        if adaptive and not np.isnan(fnr):
            # nudge tau up when we are missing less than budgeted, down when missing more
            tau = tau + gamma*(fnr - ALPHA)     # fnr>alpha -> tau up? NO: see note
            tau = float(np.clip(tau,0.0,1.0))
    return {'fnr':fnr_seq,'fpr':fpr_seq,'tau':tau_seq,
            'mean_fnr':float(np.nanmean(fnr_seq)),'mean_fpr':float(np.nanmean(fpr_seq)),
            'n_test_sz':len(test_sz)}

In [22]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

ALPHA=0.10

def static_conformal_tau(scores_pre, alpha):
    s=np.sort(scores_pre); n=len(s)
    if n==0: return 0.0
    k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
    return (s[0]-1e-9) if k==0 else s[k-1]

def train_backbone(cal_pre, cal_int):
    n_bal=min(len(cal_pre)*3,len(cal_int))
    cbi=cal_int[np.random.choice(len(cal_int),n_bal,replace=False)]
    Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
    sc=StandardScaler().fit(Xtr)
    m=LogisticRegression(max_iter=1000,class_weight='balanced').fit(sc.transform(Xtr),ytr)
    return m,sc

def run_patient(p, adaptive, gamma=0.05):
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<3: return None
    k=len(uniq); nc=max(1,int(np.ceil(k/2)))
    cal_sz, test_sz = uniq[:nc], uniq[nc:]
    cal_pre=Xp[np.isin(sid_p,cal_sz)]
    if len(cal_pre)<5: return None
    ni=len(Xi); cut=ni//2; cal_int,test_int_all=Xi[:cut],Xi[cut:]
    m,sc=train_backbone(cal_pre,cal_int)

    s_cal=m.predict_proba(sc.transform(cal_pre))[:,1]
    tau=static_conformal_tau(s_cal,ALPHA)

    s_int_test=m.predict_proba(sc.transform(test_int_all))[:,1]
    int_chunks=np.array_split(s_int_test, len(test_sz))

    fnr_seq, fpr_seq, tau_seq = [], [], []
    for j, szid in enumerate(test_sz):
        tp=Xp[sid_p==szid]
        s=m.predict_proba(sc.transform(tp))[:,1]
        fnr = np.mean(s<tau) if len(s) else np.nan
        fpr = np.mean(int_chunks[j]>=tau) if len(int_chunks[j]) else np.nan
        fnr_seq.append(fnr); fpr_seq.append(fpr); tau_seq.append(float(tau))
        if adaptive and not np.isnan(fnr):
            tau = tau - gamma*(fnr - ALPHA)      # fnr>alpha -> tau DOWN -> catch more preictal
            tau = float(np.clip(tau,0.0,1.0))
    return {'fnr':fnr_seq,'fpr':fpr_seq,'tau':tau_seq,
            'mean_fnr':float(np.nanmean(fnr_seq)),'mean_fpr':float(np.nanmean(fpr_seq)),
            'n_test_sz':len(test_sz)}

static_res, adapt_res = {}, {}
for p in cohort:
    rs=run_patient(p, adaptive=False)
    ra=run_patient(p, adaptive=True, gamma=0.05)
    if rs: static_res[p]=rs
    if ra: adapt_res[p]=ra

def summarize(res, name):
    mf=[v['mean_fnr'] for v in res.values()]
    mp=[v['mean_fpr'] for v in res.values()]
    viol=sum(f>ALPHA+0.02 for f in mf)
    print(f"\n[{name}] target FNR={ALPHA}, patients={len(mf)}")
    print(f"  mean FNR {np.nanmean(mf):.3f}, worst {np.nanmax(mf):.3f}, violations {viol}/{len(mf)}")
    print(f"  mean FPR (alarm burden) {np.nanmean(mp):.3f}")

summarize(static_res,"STATIC conformal (time-split)")
summarize(adapt_res,"ADAPTIVE conformal (time-split)")
json.dump({'static':static_res,'adaptive':adapt_res}, open(f'{OUT}/stage5_adaptive.json','w'), indent=2)
print("\nSaved stage5_adaptive.json")


[STATIC conformal (time-split)] target FNR=0.1, patients=18
  mean FNR 0.720, worst 1.000, violations 18/18
  mean FPR (alarm burden) 0.201

[ADAPTIVE conformal (time-split)] target FNR=0.1, patients=18
  mean FNR 0.695, worst 1.000, violations 18/18
  mean FPR (alarm burden) 0.205

Saved stage5_adaptive.json


In [23]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

TARGET_SENS=0.90   # catch at least 9 seizures in 10

def train_backbone(cal_pre,cal_int):
    n_bal=min(len(cal_pre)*3,len(cal_int))
    cbi=cal_int[np.random.choice(len(cal_int),n_bal,replace=False)]
    Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
    sc=StandardScaler().fit(Xtr)
    m=LogisticRegression(max_iter=1000,class_weight='balanced',C=0.5).fit(sc.transform(Xtr),ytr)
    return m,sc

def seizure_maxscore(m,sc,Xp,sid_p,sz_ids):
    """One score per seizure: the highest-scoring window in its preictal run."""
    out={}
    for szid in sz_ids:
        tp=Xp[sid_p==szid]
        if len(tp)==0: continue
        s=m.predict_proba(sc.transform(tp))[:,1]
        out[szid]=float(s.max())   # seizure caught if max >= tau
    return out

def conformal_sens_tau(cal_maxscores, target_sens):
    """Threshold on seizure-level scores rather than window-level ones.

    A seizure counts as missed when its best window still falls below tau, so
    tau sits at the alpha-quantile of the calibration seizures' peak scores."""
    alpha=1-target_sens
    s=np.sort(list(cal_maxscores.values())); n=len(s)
    if n==0: return 0.0
    k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
    return (s[0]-1e-9) if k==0 else s[k-1]

def run_patient(p, adaptive, gamma=0.05):
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<3: return None
    k=len(uniq); nc=max(2,int(np.ceil(k/2)))   # >=2 cal seizures so tau is defined
    if nc>=k: nc=k-1
    cal_sz,test_sz=uniq[:nc],uniq[nc:]
    if len(test_sz)<1: return None
    cal_pre=Xp[np.isin(sid_p,cal_sz)]
    ni=len(Xi); cut=ni//2; cal_int,test_int=Xi[:cut],Xi[cut:]
    m,sc=train_backbone(cal_pre,cal_int)

    cal_max=seizure_maxscore(m,sc,Xp,sid_p,cal_sz)
    tau=conformal_sens_tau(cal_max,TARGET_SENS)

    # false-alarm rate: share of interictal windows that clear tau (stands in for alarms/hour)
    s_int=m.predict_proba(sc.transform(test_int))[:,1]
    fa_rate=float(np.mean(s_int>=tau))

    caught=[]; tau_seq=[]
    for szid in test_sz:
        tp=Xp[sid_p==szid]
        smax=m.predict_proba(sc.transform(tp))[:,1].max() if len(tp) else 0.0
        hit=int(smax>=tau); caught.append(hit); tau_seq.append(float(tau))
        if adaptive:
            miss=1-hit
            tau=float(np.clip(tau - gamma*(miss-(1-TARGET_SENS)),0.0,1.0))
    sens=float(np.mean(caught))
    return {'sens':sens,'fa_rate':fa_rate,'n_test_sz':len(test_sz),'caught':caught}

for mode,adapt in [('STATIC',False),('ADAPTIVE',True)]:
    res={}
    for p in cohort:
        r=run_patient(p,adaptive=adapt)
        if r: res[p]=r
    sens=[v['sens'] for v in res.values()]
    fa=[v['fa_rate'] for v in res.values()]
    meets=sum(s>=TARGET_SENS-0.001 for s in sens)
    print(f"\n[{mode}] target sensitivity>={TARGET_SENS}, patients={len(sens)}")
    print(f"  mean sensitivity {np.mean(sens):.3f}, worst {np.min(sens):.3f}")
    print(f"  patients meeting guarantee: {meets}/{len(sens)}")
    print(f"  mean interictal false-alarm rate: {np.mean(fa):.3f}")
    json.dump(res,open(f'{OUT}/stage5_seizurelevel_{mode.lower()}.json','w'),indent=2)


[STATIC] target sensitivity>=0.9, patients=18
  mean sensitivity 0.426, worst 0.000
  patients meeting guarantee: 4/18
  mean interictal false-alarm rate: 0.044

[ADAPTIVE] target sensitivity>=0.9, patients=18
  mean sensitivity 0.389, worst 0.000
  patients meeting guarantee: 1/18
  mean interictal false-alarm rate: 0.041


In [24]:
import json, numpy as np, os
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.covariance import EmpiricalCovariance
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']
TARGET_SENS=0.90

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

def seizure_maxscores(score_fn, Xp, sid_p, sz_ids):
    return {sz: float(score_fn(Xp[sid_p==sz]).max()) for sz in sz_ids if (sid_p==sz).sum()>0}

def conformal_tau(cal_max_vals, target_sens):
    alpha=1-target_sens; s=np.sort(cal_max_vals); n=len(s)
    if n==0: return 0.0
    k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
    return (s[0]-1e-9) if k==0 else s[k-1]

def run_method(method):
    per_patient=[]
    fa_all=[]
    for p in cohort:
        X,y,sid,seg=load_patient(p)
        Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
        uniq=sorted(set(sid_p.tolist()))
        if len(uniq)<3: continue

        # leave-one-seizure-out: every seizure takes a turn as the held-out one
        caught=[]
        for held in uniq:
            cal_sz=[s for s in uniq if s!=held]
            cal_pre=Xp[np.isin(sid_p,cal_sz)]
            if len(cal_pre)<5: continue

            if method=='logreg_pca':
                nb=min(len(cal_pre)*3,len(Xi))
                cbi=Xi[np.random.choice(len(Xi),nb,replace=False)]
                Xtr=np.vstack([cal_pre,cbi]); ytr=np.r_[np.ones(len(cal_pre)),np.zeros(len(cbi))]
                sc=StandardScaler().fit(Xtr)
                pca=PCA(n_components=min(20,Xtr.shape[1])).fit(sc.transform(Xtr))
                m=LogisticRegression(max_iter=1000,class_weight='balanced',C=0.1).fit(pca.transform(sc.transform(Xtr)),ytr)
                score_fn=lambda A: m.predict_proba(pca.transform(sc.transform(A)))[:,1]
            elif method=='mahalanobis':
                # the anomaly model never sees a preictal window - it is fit on interictal only
                sc=StandardScaler().fit(Xi)
                cov=EmpiricalCovariance().fit(sc.transform(Xi))
                score_fn=lambda A: cov.mahalanobis(sc.transform(A))  # higher=more anomalous

            # tau from the remaining seizures' peak scores
            cal_max=seizure_maxscores(score_fn,Xp,sid_p,cal_sz)
            tau=conformal_tau(list(cal_max.values()),TARGET_SENS)
            # did the held-out seizure clear it?
            tp=Xp[sid_p==held]
            hit=int(score_fn(tp).max()>=tau) if len(tp) else 0
            caught.append(hit)
            # false-alarm rate on an interictal sample
            si=score_fn(Xi[np.random.choice(len(Xi),min(500,len(Xi)),replace=False)])
            fa_all.append(np.mean(si>=tau))
        if caught:
            per_patient.append(np.mean(caught))
    return per_patient, fa_all

for method in ['logreg_pca','mahalanobis']:
    sens,fa=run_method(method)
    meets=sum(s>=TARGET_SENS-0.001 for s in sens)
    print(f"\n[{method} | LOSO] target sens>={TARGET_SENS}, patients={len(sens)}")
    print(f"  mean sensitivity {np.mean(sens):.3f}, worst {np.min(sens):.3f}")
    print(f"  meeting guarantee: {meets}/{len(sens)}, mean FA rate {np.mean(fa):.3f}")


[logreg_pca | LOSO] target sens>=0.9, patients=18
  mean sensitivity 0.517, worst 0.000
  meeting guarantee: 0/18, mean FA rate 0.039

[mahalanobis | LOSO] target sens>=0.9, patients=18
  mean sensitivity 0.777, worst 0.667
  meeting guarantee: 1/18, mean FA rate 0.085


In [25]:
import json, numpy as np, os
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
np.random.seed(42)
OUT='/kaggle/working/artifacts'
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

def seizure_agg(scores, agg='max', topk=3):
    if len(scores)==0: return -np.inf
    if agg=='max': return float(np.max(scores))
    if agg=='topk': return float(np.mean(np.sort(scores)[-topk:]))
    if agg=='persist':  # max over 3-consecutive-window means
        if len(scores)<3: return float(np.max(scores))
        return float(np.max([scores[i:i+3].mean() for i in range(len(scores)-2)]))

def conformal_tau(vals, target_sens):
    alpha=1-target_sens; s=np.sort(vals); n=len(s)
    if n==0: return -np.inf
    k=int(np.floor(alpha*(n+1))); k=min(max(k,0),n)
    return (s[0]-1e-9) if k==0 else s[k-1]

def run(agg, target_cal):
    """Sweep the seizure-level aggregation and the calibration target.

    target_cal is what we calibrate against, which can sit above 0.9 to buy
    some headroom over the sensitivity we actually want."""
    sens_list, fa_list = [], []
    for p in cohort:
        X,y,sid,seg=load_patient(p)
        Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
        uniq=sorted(set(sid_p.tolist()))
        if len(uniq)<3: continue
        caught=[]; fap=[]
        for held in uniq:
            cal_sz=[s for s in uniq if s!=held]
            sc=StandardScaler().fit(Xi)
            cov=EmpiricalCovariance().fit(sc.transform(Xi))
            score_fn=lambda A: cov.mahalanobis(sc.transform(A))
            cal_vals=[seizure_agg(score_fn(Xp[sid_p==s]),agg) for s in cal_sz]
            tau=conformal_tau(cal_vals,target_cal)
            tp=Xp[sid_p==held]
            caught.append(int(seizure_agg(score_fn(tp),agg)>=tau))
            si=score_fn(Xi[np.random.choice(len(Xi),min(500,len(Xi)),replace=False)])
            fap.append(np.mean(si>=tau))
        if caught:
            sens_list.append(np.mean(caught)); fa_list.append(np.mean(fap))
    return sens_list, fa_list

for agg in ['max','topk','persist']:
    for tcal in [0.90, 0.95, 0.99]:
        sens,fa=run(agg,tcal)
        meets=sum(s>=0.90-0.001 for s in sens)
        print(f"[agg={agg:7s} cal={tcal}] mean_sens {np.mean(sens):.3f}  worst {np.min(sens):.3f}  meets>=0.9 {meets}/{len(sens)}  FA {np.mean(fa):.3f}")

[agg=max     cal=0.9] mean_sens 0.777  worst 0.667  meets>=0.9 1/18  FA 0.070
[agg=max     cal=0.95] mean_sens 0.777  worst 0.667  meets>=0.9 1/18  FA 0.070
[agg=max     cal=0.99] mean_sens 0.777  worst 0.667  meets>=0.9 1/18  FA 0.072
[agg=topk    cal=0.9] mean_sens 0.777  worst 0.667  meets>=0.9 1/18  FA 0.083
[agg=topk    cal=0.95] mean_sens 0.777  worst 0.667  meets>=0.9 1/18  FA 0.084
[agg=topk    cal=0.99] mean_sens 0.777  worst 0.667  meets>=0.9 1/18  FA 0.087
[agg=persist cal=0.9] mean_sens 0.777  worst 0.667  meets>=0.9 1/18  FA 0.107
[agg=persist cal=0.95] mean_sens 0.777  worst 0.667  meets>=0.9 1/18  FA 0.108
[agg=persist cal=0.99] mean_sens 0.777  worst 0.667  meets>=0.9 1/18  FA 0.107


In [26]:
import json, numpy as np, os
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
np.random.seed(42)
OUT='/kaggle/working/artifacts'; os.makedirs(f'{OUT}/figures',exist_ok=True)
cohort=json.load(open(f'{OUT}/cohort.json'))['cohort']

def load_patient(p):
    d=np.load(f'{OUT}/{p}_features.npz',allow_pickle=True)
    return d['X'],d['y'],d['seizure_id'],d['seg_type']

# Step 4 - per-seizure separability. For each seizure, how far does its
# preictal anomaly score sit above that patient's own interictal spread?
records=[]
for p in cohort:
    X,y,sid,seg=load_patient(p)
    Xp,Xi=X[y==1],X[y==0]; sid_p=sid[y==1]
    uniq=sorted(set(sid_p.tolist()))
    if len(uniq)<3: continue
    sc=StandardScaler().fit(Xi)
    cov=EmpiricalCovariance().fit(sc.transform(Xi))
    int_scores=cov.mahalanobis(sc.transform(Xi))
    int_p95=np.percentile(int_scores,95)   # interictal 95th pct = natural alarm level
    for szid in uniq:
        pre=cov.mahalanobis(sc.transform(Xp[sid_p==szid]))
        # share of this seizure's preictal windows that clear the interictal 95th percentile
        detectable=float(np.mean(pre>int_p95))
        records.append({'patient':p,'seizure':szid,
                        'pre_max':float(pre.max()),'int_p95':float(int_p95),
                        'sep_ratio':float(pre.max()/int_p95),
                        'detectable_frac':detectable})

json.dump(records,open(f'{OUT}/ceiling_analysis.json','w'),indent=2)
sep=np.array([r['sep_ratio'] for r in records])
det=np.array([r['detectable_frac'] for r in records])
print(f"Total seizures analyzed: {len(records)}")
print(f"Seizures where preictal max EXCEEDS interictal-95 (sep_ratio>1): {np.mean(sep>1)*100:.1f}%")
print(f"Seizures essentially undetectable (detectable_frac<0.05): {np.mean(det<0.05)*100:.1f}%")
print(f"sep_ratio: median {np.median(sep):.2f}, 25th {np.percentile(sep,25):.2f}, 75th {np.percentile(sep,75):.2f}")

# distribution of separability across every seizure in the cohort
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].hist(sep,bins=30,color='#c0392b',edgecolor='white')
ax[0].axvline(1.0,color='black',ls='--',lw=1.5,label='detectability threshold')
ax[0].set_xlabel('Separability ratio (preictal max / interictal 95th pct)')
ax[0].set_ylabel('Number of seizures'); ax[0].set_title('Per-seizure detectability'); ax[0].legend()
ax[1].hist(det,bins=30,color='#2c3e50',edgecolor='white')
ax[1].set_xlabel('Fraction of preictal windows above interictal-95')
ax[1].set_ylabel('Number of seizures'); ax[1].set_title('Detectable preictal mass per seizure')
plt.tight_layout()
plt.savefig(f'{OUT}/figures/ceiling_separability.png',dpi=300,bbox_inches='tight')
plt.savefig(f'{OUT}/figures/ceiling_separability.pdf',bbox_inches='tight')
print(f"\nSaved figure: {OUT}/figures/ceiling_separability.png (+.pdf)")

Total seizures analyzed: 95
Seizures where preictal max EXCEEDS interictal-95 (sep_ratio>1): 90.5%
Seizures essentially undetectable (detectable_frac<0.05): 16.8%
sep_ratio: median 3.01, 25th 1.37, 75th 7.67

Saved figure: /kaggle/working/artifacts/figures/ceiling_separability.png (+.pdf)
